# 05 — Model Explainability
Inspect the fitted Logistic Regression coefficients to understand which encoded workforce features influence predicted attrition risk.

This notebook provides global coefficient-based explainability; SHAP can be added after the final model pipeline is frozen.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
df=pd.read_csv(Path('../data/raw/employee_attrition.csv'))
y=df.pop('Attrition').map({'No':0,'Yes':1})
X=df.drop(columns=['EmployeeNumber','EmployeeCount','Over18','StandardHours'])
cat=X.select_dtypes(include=['object','string']).columns; num=X.select_dtypes(exclude=['object','string']).columns
prep=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
pipe=Pipeline([('prep',prep),('model',LogisticRegression(class_weight='balanced',max_iter=2000))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
pipe.fit(X_train,y_train)
names=pipe.named_steps['prep'].get_feature_names_out()
coef=pipe.named_steps['model'].coef_[0]
importance=pd.DataFrame({'feature':names,'coefficient':coef,'abs_coefficient':abs(coef)}).sort_values('abs_coefficient',ascending=False)
importance.head(20)